# EDA: E-commerce Fraud Data (Fraud_Data.csv)

Exploratory analysis of e-commerce transaction dataset with geolocation enrichment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load Data

In [ ]:
# Load datasets
fraud_df = pd.read_csv('../data/Fraud_Data.csv')
ip_country_df = pd.read_csv('../data/IpAddress_to_Country.csv')

print("Fraud Data Shape:", fraud_df.shape)
print("\nIP Country Data Shape:", ip_country_df.shape)
print("\nFraud Data Columns:", fraud_df.columns.tolist())
print("\nIP Country Columns:", ip_country_df.columns.tolist())

In [ ]:
# Display first few rows
print("Fraud Data Sample:")
print(fraud_df.head())
print("\n" + "="*80)
print("\nIP Country Sample:")
print(ip_country_df.head())

## 2. Data Quality Assessment

In [ ]:
# Data types and missing values
print("Data Types and Missing Values:")
print(fraud_df.info())
print("\nMissing Values Summary:")
missing = fraud_df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

In [ ]:
# Check for duplicates
print(f"Total Records: {len(fraud_df)}")
print(f"Duplicate Records: {fraud_df.duplicated().sum()}")
print(f"Unique Users: {fraud_df['user_id'].nunique()}")
print(f"Unique Devices: {fraud_df['device_id'].nunique()}")
print(f"Unique IP Addresses: {fraud_df['ip_address'].nunique()}")

## 3. Target Variable Analysis

In [ ]:
# Class distribution
class_counts = fraud_df['class'].value_counts()
class_pct = fraud_df['class'].value_counts(normalize=True) * 100

print("Target Variable Distribution:")
print(f"Legitimate (0): {class_counts[0]} ({class_pct[0]:.2f}%)")
print(f"Fraud (1): {class_counts[1]} ({class_pct[1]:.2f}%)")
print(f"\nImbalance Ratio: {class_counts[0] / class_counts[1]:.2f}:1")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fraud_df['class'].value_counts().plot(kind='bar', ax=ax1, color=['green', 'red'])
ax1.set_title('Fraud Distribution (Count)', fontsize=12)
ax1.set_xlabel('Class (0=Legitimate, 1=Fraud)')
ax1.set_ylabel('Count')

fraud_df['class'].value_counts(normalize=True).plot(kind='pie', ax=ax2, 
                                                       labels=['Legitimate', 'Fraud'],
                                                       autopct='%1.2f%%',
                                                       colors=['green', 'red'])
ax2.set_title('Fraud Distribution (%)', fontsize=12)
ax2.set_ylabel('')
plt.tight_layout()
plt.savefig('../notebooks/fraud_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Temporal Analysis

In [ ]:
# Convert to datetime
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

# Calculate time-since-signup
fraud_df['time_since_signup'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600  # hours

print("Time-Since-Signup Statistics:")
print(fraud_df['time_since_signup'].describe())
print(f"\nMin: {fraud_df['time_since_signup'].min()} hours")
print(f"Max: {fraud_df['time_since_signup'].max()} hours")

In [ ]:
# Extract temporal features
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek
fraud_df['is_weekend'] = (fraud_df['day_of_week'] >= 5).astype(int)

# Time-since-signup by class
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

fraud_df[fraud_df['class'] == 0]['time_since_signup'].hist(bins=50, ax=axes[0], color='green', alpha=0.7)
axes[0].set_title('Time-Since-Signup: Legitimate Transactions', fontsize=12)
axes[0].set_xlabel('Hours')
axes[0].set_ylabel('Frequency')

fraud_df[fraud_df['class'] == 1]['time_since_signup'].hist(bins=50, ax=axes[1], color='red', alpha=0.7)
axes[1].set_title('Time-Since-Signup: Fraudulent Transactions', fontsize=12)
axes[1].set_xlabel('Hours')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../notebooks/time_since_signup_dist.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTime-Since-Signup by Class:")
print(fraud_df.groupby('class')['time_since_signup'].describe())

## 5. Geographic Analysis (IP to Country)

In [ ]:
def ip_to_integer(ip_str):
    """Convert IP address string to integer"""
    parts = ip_str.split('.')
    return int(parts[0]) * 256**3 + int(parts[1]) * 256**2 + int(parts[2]) * 256 + int(parts[3])

# Convert IP addresses to integers
print("Converting IP addresses to integers...")
fraud_df['ip_integer'] = fraud_df['ip_address'].apply(ip_to_integer)
ip_country_df['lower_bound_integer'] = ip_country_df['lower_bound_ip_address'].apply(ip_to_integer)
ip_country_df['upper_bound_integer'] = ip_country_df['upper_bound_ip_address'].apply(ip_to_integer)

print("Sample IP to Integer Conversion:")
print(fraud_df[['ip_address', 'ip_integer']].head())

In [ ]:
# Range-based IP to Country lookup
def lookup_country(ip_int, ip_country_df):
    """Lookup country for an IP address using binary search"""
    # Find rows where ip_int is between lower and upper bounds
    match = ip_country_df[
        (ip_country_df['lower_bound_integer'] <= ip_int) & 
        (ip_country_df['upper_bound_integer'] >= ip_int)
    ]
    if len(match) > 0:
        return match.iloc[0]['country']
    return 'Unknown'

# Apply lookup (this may take a moment)
print("Looking up countries for IP addresses...")
fraud_df['country'] = fraud_df['ip_integer'].apply(
    lambda x: lookup_country(x, ip_country_df)
)

print(f"\nCountries Found: {fraud_df['country'].nunique()}")
print(f"Unknown: {(fraud_df['country'] == 'Unknown').sum()}")
print("\nTop 10 Countries:")
print(fraud_df['country'].value_counts().head(10))

In [ ]:
# Fraud rate by country
fraud_by_country = fraud_df.groupby('country').agg({
    'class': ['sum', 'count']
}).round(4)
fraud_by_country.columns = ['fraud_count', 'total_transactions']
fraud_by_country['fraud_rate'] = (fraud_by_country['fraud_count'] / fraud_by_country['total_transactions'] * 100).round(2)
fraud_by_country = fraud_by_country.sort_values('fraud_rate', ascending=False)

print("\nTop 15 Countries by Fraud Rate:")
print(fraud_by_country.head(15))

## 6. Categorical Features Analysis

In [ ]:
# Browser analysis
print("Browser Distribution:")
print(fraud_df['browser'].value_counts().head(10))

# Browser fraud rate
browser_fraud = fraud_df.groupby('browser')['class'].agg(['sum', 'count'])
browser_fraud['fraud_rate'] = (browser_fraud['sum'] / browser_fraud['count'] * 100).round(2)
browser_fraud = browser_fraud.sort_values('fraud_rate', ascending=False)
print("\nTop 10 Browsers by Fraud Rate:")
print(browser_fraud.head(10))

In [ ]:
# Source analysis
print("Source (Marketing Channel) Distribution:")
print(fraud_df['source'].value_counts())

# Source fraud rate
source_fraud = fraud_df.groupby('source')['class'].agg(['sum', 'count'])
source_fraud['fraud_rate'] = (source_fraud['sum'] / source_fraud['count'] * 100).round(2)
print("\nFraud Rate by Source:")
print(source_fraud)

In [ ]:
# Sex analysis
print("Sex Distribution:")
print(fraud_df['sex'].value_counts())

# Sex fraud rate
sex_fraud = fraud_df.groupby('sex')['class'].agg(['sum', 'count'])
sex_fraud['fraud_rate'] = (sex_fraud['sum'] / sex_fraud['count'] * 100).round(2)
print("\nFraud Rate by Sex:")
print(sex_fraud)

## 7. Numerical Features Analysis

In [ ]:
# Purchase value analysis
print("Purchase Value Statistics:")
print(fraud_df['purchase_value'].describe())

# By class
print("\nPurchase Value by Class:")
print(fraud_df.groupby('class')['purchase_value'].describe())

In [ ]:
# Age analysis
print("Age Statistics:")
print(fraud_df['age'].describe())

# By class
print("\nAge by Class:")
print(fraud_df.groupby('class')['age'].describe())

In [ ]:
# Visualize numerical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Purchase value
fraud_df[fraud_df['class'] == 0]['purchase_value'].hist(bins=50, ax=axes[0, 0], 
                                                          color='green', alpha=0.7, label='Legitimate')
fraud_df[fraud_df['class'] == 1]['purchase_value'].hist(bins=50, ax=axes[0, 0], 
                                                          color='red', alpha=0.7, label='Fraud')
axes[0, 0].set_title('Purchase Value Distribution', fontsize=12)
axes[0, 0].set_xlabel('Value ($)')
axes[0, 0].legend()

# Age
fraud_df[fraud_df['class'] == 0]['age'].hist(bins=50, ax=axes[0, 1], 
                                               color='green', alpha=0.7, label='Legitimate')
fraud_df[fraud_df['class'] == 1]['age'].hist(bins=50, ax=axes[0, 1], 
                                               color='red', alpha=0.7, label='Fraud')
axes[0, 1].set_title('Age Distribution', fontsize=12)
axes[0, 1].set_xlabel('Age (years)')
axes[0, 1].legend()

# Hour of day
hour_fraud = fraud_df.groupby('hour_of_day')['class'].mean() * 100
axes[1, 0].plot(hour_fraud.index, hour_fraud.values, marker='o', linewidth=2, markersize=6)
axes[1, 0].set_title('Fraud Rate by Hour of Day', fontsize=12)
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Fraud Rate (%)')
axes[1, 0].grid(True, alpha=0.3)

# Day of week
day_fraud = fraud_df.groupby('day_of_week')['class'].mean() * 100
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 1].bar(range(7), day_fraud.values, color='steelblue', alpha=0.7)
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(days)
axes[1, 1].set_title('Fraud Rate by Day of Week', fontsize=12)
axes[1, 1].set_ylabel('Fraud Rate (%)')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../notebooks/numerical_features_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Feature Correlations

In [ ]:
# Create numeric features for correlation
fraud_df_numeric = fraud_df[['purchase_value', 'age', 'hour_of_day', 'day_of_week', 
                               'is_weekend', 'time_since_signup', 'class']].copy()

# Correlation matrix
corr_matrix = fraud_df_numeric.corr()

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('../notebooks/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nCorrelation with Target (class):")
print(corr_matrix['class'].sort_values(ascending=False))

## 9. Save Processed Data

In [ ]:
# Save processed data
fraud_df.to_csv('../data/processed/fraud_data_processed.csv', index=False)
print("Saved: fraud_data_processed.csv")
print(f"\nFinal shape: {fraud_df.shape}")
print(f"Columns: {fraud_df.columns.tolist()}")